In [3]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd().parent
os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("Current working directory:", Path.cwd())

Project root: /home/syed/GWU/Spring 2026/Big Data Analytics/renewable-energy-forecasting-pipeline
Current working directory: /home/syed/GWU/Spring 2026/Big Data Analytics/renewable-energy-forecasting-pipeline


In [4]:
from pyspark.sql import functions as F

from src.common.spark_utils import get_local_spark_session, stop_spark_session
from src.parsing.parse_all_fields import add_all_parsed_weather_columns

In [5]:
spark = get_local_spark_session("parse-validation")

spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/17 22:39:40 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/17 22:39:43 WARN FileSystem: Cannot load filesystem
java.util.ServiceConfigurationError: org.apache.hadoop.fs.FileSystem: Provider org.apache.hadoop.fs.viewfs.ViewFileSystem could not be instantiated
	at java.base/java.util.ServiceLoader.fail(ServiceLoader.java:552)
	at java.base/java.util.ServiceLoader$ProviderImpl.newInstance(ServiceLoader.java:712)
	at java.base/java.util.ServiceLoader$ProviderImpl.get(ServiceLoader.java:672)
	at java.base/java.util.ServiceLoader$2.next(ServiceLoader.java:1256)
	at org.apache.hadoop.fs.FileSystem.loadFileSystems(FileSystem.java:3525)
	at org.apache.hadoop.fs.FileSystem.getFileSystemClass(FileSyst

In [6]:
sample_data = [
    (
        "324,1,H,0051,1",
        "+0093,1",
        "+0078,1",
        "10132,1",
        "016093,1,N,1",
        "02200,1,5,0",
    ),
    (
        "999,1,H,9999,1",
        "+9999,1",
        "-9999,1",
        "99999,1",
        "999999,1,9,1",
        "99999,1,9,9",
    ),
    (
        None,
        None,
        None,
        None,
        None,
        None,
    ),
    (
        "bad,data",
        "bad,data",
        "bad,data",
        "bad,data",
        "bad,data",
        "bad,data",
    ),
]

raw_df = spark.createDataFrame(
    sample_data,
    ["WND", "TMP", "DEW", "SLP", "VIS", "CIG"],
)

raw_df.show(truncate=False, vertical=True)

-RECORD 0-------------
 WND | 324,1,H,0051,1 
 TMP | +0093,1        
 DEW | +0078,1        
 SLP | 10132,1        
 VIS | 016093,1,N,1   
 CIG | 02200,1,5,0    
-RECORD 1-------------
 WND | 999,1,H,9999,1 
 TMP | +9999,1        
 DEW | -9999,1        
 SLP | 99999,1        
 VIS | 999999,1,9,1   
 CIG | 99999,1,9,9    
-RECORD 2-------------
 WND | NULL           
 TMP | NULL           
 DEW | NULL           
 SLP | NULL           
 VIS | NULL           
 CIG | NULL           
-RECORD 3-------------
 WND | bad,data       
 TMP | bad,data       
 DEW | bad,data       
 SLP | bad,data       
 VIS | bad,data       
 CIG | bad,data       



In [7]:
parsed_df = add_all_parsed_weather_columns(raw_df)
parsed_df.show(truncate=False, vertical=True)

-RECORD 0------------------------------------
 WND                        | 324,1,H,0051,1 
 TMP                        | +0093,1        
 DEW                        | +0078,1        
 SLP                        | 10132,1        
 VIS                        | 016093,1,N,1   
 CIG                        | 02200,1,5,0    
 wind_direction_degrees     | 324            
 wind_direction_qc          | 1              
 wind_observation_type      | H              
 wind_speed_ms              | 5.1            
 wind_speed_qc              | 1              
 temperature_c              | 9.300000       
 temperature_qc             | 1              
 dew_point_c                | 7.800000       
 dew_point_qc               | 1              
 sea_level_pressure_hpa     | 1013.200000    
 sea_level_pressure_qc      | 1              
 visibility_distance_m      | 16093.0        
 visibility_distance_qc     | 1              
 visibility_variability     | N              
 visibility_variability_qc  | 1   

In [8]:
parsed_df.printSchema()

root
 |-- WND: string (nullable = true)
 |-- TMP: string (nullable = true)
 |-- DEW: string (nullable = true)
 |-- SLP: string (nullable = true)
 |-- VIS: string (nullable = true)
 |-- CIG: string (nullable = true)
 |-- wind_direction_degrees: integer (nullable = true)
 |-- wind_direction_qc: string (nullable = true)
 |-- wind_observation_type: string (nullable = true)
 |-- wind_speed_ms: double (nullable = true)
 |-- wind_speed_qc: string (nullable = true)
 |-- temperature_c: decimal(17,6) (nullable = true)
 |-- temperature_qc: string (nullable = true)
 |-- dew_point_c: decimal(17,6) (nullable = true)
 |-- dew_point_qc: string (nullable = true)
 |-- sea_level_pressure_hpa: decimal(17,6) (nullable = true)
 |-- sea_level_pressure_qc: string (nullable = true)
 |-- visibility_distance_m: decimal(13,1) (nullable = true)
 |-- visibility_distance_qc: string (nullable = true)
 |-- visibility_variability: string (nullable = true)
 |-- visibility_variability_qc: string (nullable = true)
 |-- ce

In [9]:
parsed_df.select(
    "wind_direction_degrees",
    "wind_direction_qc",
    "wind_observation_type",
    "wind_speed_ms",
    "wind_speed_qc",
    "temperature_c",
    "temperature_qc",
    "dew_point_c",
    "dew_point_qc",
    "sea_level_pressure_hpa",
    "sea_level_pressure_qc",
    "visibility_distance_m",
    "visibility_distance_qc",
    "visibility_variability",
    "visibility_variability_qc",
    "ceiling_height_m",
    "ceiling_height_qc",
    "ceiling_determination_code",
    "ceiling_cavok",
).show(truncate=False, vertical=True)

-RECORD 0---------------------------------
 wind_direction_degrees     | 324         
 wind_direction_qc          | 1           
 wind_observation_type      | H           
 wind_speed_ms              | 5.1         
 wind_speed_qc              | 1           
 temperature_c              | 9.300000    
 temperature_qc             | 1           
 dew_point_c                | 7.800000    
 dew_point_qc               | 1           
 sea_level_pressure_hpa     | 1013.200000 
 sea_level_pressure_qc      | 1           
 visibility_distance_m      | 16093.0     
 visibility_distance_qc     | 1           
 visibility_variability     | N           
 visibility_variability_qc  | 1           
 ceiling_height_m           | 2200.0      
 ceiling_height_qc          | 1           
 ceiling_determination_code | 5           
 ceiling_cavok              | 0           
-RECORD 1---------------------------------
 wind_direction_degrees     | NULL        
 wind_direction_qc          | 1           
 wind_obser

In [10]:
parsed_df.select(
    F.count(F.lit(1)).alias("row_count"),
    F.sum(F.col("wind_direction_degrees").isNull().cast("int")).alias("wind_direction_degrees_nulls"),
    F.sum(F.col("wind_speed_ms").isNull().cast("int")).alias("wind_speed_ms_nulls"),
    F.sum(F.col("temperature_c").isNull().cast("int")).alias("temperature_c_nulls"),
    F.sum(F.col("dew_point_c").isNull().cast("int")).alias("dew_point_c_nulls"),
    F.sum(F.col("sea_level_pressure_hpa").isNull().cast("int")).alias("sea_level_pressure_hpa_nulls"),
    F.sum(F.col("visibility_distance_m").isNull().cast("int")).alias("visibility_distance_m_nulls"),
    F.sum(F.col("ceiling_height_m").isNull().cast("int")).alias("ceiling_height_m_nulls"),
).show(truncate=False, vertical=True)

-RECORD 0---------------------------
 row_count                    | 4   
 wind_direction_degrees_nulls | 3   
 wind_speed_ms_nulls          | 3   
 temperature_c_nulls          | 3   
 dew_point_c_nulls            | 3   
 sea_level_pressure_hpa_nulls | 3   
 visibility_distance_m_nulls  | 3   
 ceiling_height_m_nulls       | 3   



In [11]:
parsed_df.select(
    F.min("wind_direction_degrees").alias("min_wind_direction_degrees"),
    F.max("wind_direction_degrees").alias("max_wind_direction_degrees"),
    F.min("wind_speed_ms").alias("min_wind_speed_ms"),
    F.max("wind_speed_ms").alias("max_wind_speed_ms"),
    F.min("temperature_c").alias("min_temperature_c"),
    F.max("temperature_c").alias("max_temperature_c"),
    F.min("dew_point_c").alias("min_dew_point_c"),
    F.max("dew_point_c").alias("max_dew_point_c"),
    F.min("sea_level_pressure_hpa").alias("min_sea_level_pressure_hpa"),
    F.max("sea_level_pressure_hpa").alias("max_sea_level_pressure_hpa"),
    F.min("visibility_distance_m").alias("min_visibility_distance_m"),
    F.max("visibility_distance_m").alias("max_visibility_distance_m"),
    F.min("ceiling_height_m").alias("min_ceiling_height_m"),
    F.max("ceiling_height_m").alias("max_ceiling_height_m"),
).show(truncate=False, vertical=True)

-RECORD 0---------------------------------
 min_wind_direction_degrees | 324         
 max_wind_direction_degrees | 324         
 min_wind_speed_ms          | 5.1         
 max_wind_speed_ms          | 5.1         
 min_temperature_c          | 9.300000    
 max_temperature_c          | 9.300000    
 min_dew_point_c            | 7.800000    
 max_dew_point_c            | 7.800000    
 min_sea_level_pressure_hpa | 1013.200000 
 max_sea_level_pressure_hpa | 1013.200000 
 min_visibility_distance_m  | 16093.0     
 max_visibility_distance_m  | 16093.0     
 min_ceiling_height_m       | 2200.0      
 max_ceiling_height_m       | 2200.0      



---
## Parsing Validation Observations

* Core NOAA ISD fields (WND, TMP, DEW, SLP, VIS, CIG) were successfully parsed into structured numeric and categorical columns.
* Sentinel values (e.g., 999, 9999, +9999, 99999, 999999) are correctly converted to NULL.
* Null input rows propagate cleanly to NULL outputs.
* Malformed rows (e.g., `"bad,data"`) do not break the pipeline and result in NULL numeric values.
* Some malformed QC values (e.g., `"data"`) are retained in QC columns for further validation in downstream layers.
* Parsed numeric fields are represented as `decimal` types in Spark rather than `double`, which is acceptable but should be standardized in future layers if needed.
* All numeric outputs fall within realistic physical ranges based on domain expectations.